# Data Validation and Quality Assurance

## Objective
Validate the cleaned enterprise datasets for referential integrity, schema completeness, numeric correctness, and minimum data volume before the platform moves into exploratory analysis and feature engineering.

The reusable validation engine lives in `backend/app/pipelines/data_validation.py`.

## Business Context

Enterprise analytics pipelines fail silently when source data drifts. This notebook makes the quality gates explicit so future forecasting, segmentation, and recommendation notebooks operate on trusted datasets.

## Architecture and Implementation Plan

1. Load the processed dataset layer produced by the cleaning module.
2. Run table-level schema, primary key, foreign key, numeric range, and row-count checks.
3. Persist a machine-readable validation report and summary artifact.
4. Visualize pass and fail counts for quick review.
5. Summarize quality posture and next steps for downstream modeling.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
backend_root = project_root / 'backend'
sys.path.insert(0, str(backend_root))

from app.pipelines.data_validation import EnterpriseDataValidator, ValidationPaths

processed_dir = project_root / 'processed'
reports_dir = project_root / 'reports'
validator = EnterpriseDataValidator(ValidationPaths(processed_dir=processed_dir, reports_dir=reports_dir))
validator.EXPECTED_MIN_ROWS = {
    'customers': 1,
    'products': 1,
    'suppliers': 1,
    'employees': 1,
    'marketing_campaigns': 1,
    'orders': 1,
    'inventory_snapshots': 1,
    'finance_monthly': 1,
    'operations_daily': 1,
    'customer_kpis': 1,
}
validator

## Run Validation

The next cell executes the quality checks and writes the validation report to `reports/`.

In [ ]:
validation_report = validator.validate_all()
validation_report.head(20)

## Validation Summary

Review the pass and fail counts together with the computed quality score.

In [ ]:
summary = validation_report.groupby('status').size().reindex(['pass', 'fail', 'skip'], fill_value=0)
quality_score = validator.quality_score(validation_report)
print(f'Quality score: {quality_score}%')
summary

## Visualization

The chart below makes it easier to spot whether the quality gates are passing at a glance.

In [ ]:
ax = summary.loc[['pass', 'fail', 'skip']].plot(kind='bar', color=['#1f77b4', '#d62728', '#7f7f7f'], figsize=(8, 4), title='Validation Status Counts')
ax.set_xlabel('Status')
ax.set_ylabel('Check Count')
plt.tight_layout()
plt.show()

## Failed Checks Review

If any checks fail, inspect them here before moving on to EDA and feature engineering.

In [ ]:
validation_report.loc[validation_report['status'] == 'fail'].sort_values(['table_name', 'check_name'])

## Summary

The validated enterprise datasets are now ready for exploratory analysis and model feature construction. The report artifacts in `reports/` provide traceable evidence of data quality for the backend and future automation.